## **Import All Dependencies** 

In [10]:
# Standard library
import os
from pathlib import Path

# Third-party libraries
import dask.dataframe as dd
import duckdb
import pandas as pd
import pyarrow.dataset as ds
import matplotlib as mpl
import matplotlib.pyplot as plt


## **Read File Path** 


In [7]:
# ─── CONFIG ────────────────────────────────────────────────────────────────────
PARQUET_PATH = Path('/Users/chromatrical/CAREER/Local Linkedin DB/DataBase/USA.parquet')
OUTPUT_PATH  = PARQUET_PATH.with_name("USA_filtered.parquet")

cols_to_check = ["Industry", "Industry 2", "Company Industry"]
# ────────────────────────────────────────────────────────────────────────────────

### **Check Original Data**

```Rows: 73,800,000, Columns: 3```

In [8]:
import dask.dataframe as dd

PARQUET_PATH = "/Users/chromatrical/CAREER/Local Linkedin DB/DataBase/USA.parquet"
cols = ["Industry", "Industry 2", "Company Industry"]

# lazy load only those columns
ddf = dd.read_parquet(PARQUET_PATH, columns=cols)

# .head() pulls just the first N rows into memory
preview = ddf.head(10)
preview

,Industry,Industry 2,Company Industry
0,non-profit organization management,<NA>,<NA>
1,<NA>,<NA>,<NA>
2,hospitality,<NA>,primary/secondary education
3,<NA>,<NA>,<NA>
4,<NA>,education,<NA>
5,entertainment,<NA>,<NA>
6,non-profit organization management,<NA>,<NA>
7,<NA>,<NA>,<NA>
8,"health, wellness and fitness",<NA>,<NA>
9,<NA>,<NA>,<NA>


In [ ]:
ddf = dd.read_parquet(PARQUET_PATH, columns=cols_to_check)
print(ddf.shape)
n_rows = ddf.shape[0].compute()
n_cols = ddf.shape[1]            # already an int
print(f"Rows: {n_rows:,}, Columns: {n_cols}")

(<dask_expr.expr.Scalar: expr=ReadParquetFSSpec(b0c4899).size() // 3, dtype=int64>, 3)
Rows: 73,800,000, Columns: 3


### **Drop N/A, and output filtered data referencing:**
- `"Industry", "Industry 2", "Company Industry"`
```File size: 16.27 GB
Shape: 51,352,619 rows × 62 columns
```

In [ ]:
import duckdb
from pathlib import Path

# ─── CONFIG ────────────────────────────────────────────────────────────────────
IN_PATH  = Path("/Users/chromatrical/CAREER/Local Linkedin DB/DataBase/USA.parquet")
OUT_PATH = IN_PATH.with_name("USA_db.parquet")
CHECK_COLS = ["Industry", "Industry 2", "Company Industry"]
# ────────────────────────────────────────────────────────────────────────────────

con = duckdb.connect()

# 1) For each filter column, gather the "valid" values (count >= 50)
valid = {}
for col in CHECK_COLS:
    # Double-quote the column name in SQL
    col_q = f'"{col}"'
    sql = f"""
      SELECT {col_q}
      FROM '{IN_PATH}'
      WHERE {col_q} IS NOT NULL
      GROUP BY {col_q}
      HAVING COUNT(*) >= 50
    """
    rows = con.execute(sql).fetchall()
    # Unpack the 1-tuples into a Python list
    valid[col] = [row[0] for row in rows]

# 2) Build the filter clause, with each value properly SQL‐escaped
#    We use Python repr() here to get a quoted string literal for each value
in_clauses = []
for col in CHECK_COLS:
    col_q = f'"{col}"'
    # build e.g.  "Industry" IN ('Retail','Manufacturing',...)
    vals = valid[col]
    if not vals:
        # no valid values → this clause will never match
        in_clauses.append("FALSE")
    else:
        list_sql = ",".join(repr(v) for v in vals)
        in_clauses.append(f"{col_q} IN ({list_sql})")

# Join with OR
in_filter = " OR ".join(in_clauses)

# 3) Copy out with both filters in one pass
sql_copy = f"""
  COPY (
    SELECT *
    FROM '{IN_PATH}'
    WHERE NOT (
       "Industry" IS NULL
       AND "Industry 2" IS NULL
       AND "Company Industry" IS NULL
    )
    AND ({in_filter})
  )
  TO '{OUT_PATH}'
  (FORMAT PARQUET)
"""
con.execute(sql_copy)

print(f"Wrote filtered Parquet to {OUT_PATH}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote filtered Parquet to /Users/chromatrical/CAREER/Local Linkedin DB/DataBase/USA_filtered.parquet


### **EDA & Data Visualization**

In [41]:
import duckdb
import pandas as pd
import os
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import pyarrow.dataset as ds
import matplotlib as mpl

# ─── CONFIG ────────────────────────────────────────────────────────────────────
PARQUET_PATH = '/Users/chromatrical/CAREER/Local Linkedin DB/DataBase/USA_filtered.parquet'
highlight_cols = {
    "Job title", "Industry", "Industry 2", "Company Industry", "Company Size",
    "Company Website", "Emails", "Company Linkedin Url", "Company Name",
    "First Name", "Last Name", "Linkedin Url", "Full Name"
}


# Now this will work, because ds is the module
dataset = ds.dataset(str(PARQUET_PATH), format="parquet")
schema = dataset.schema
n_cols = len(schema.names)


# ds = ds.dataset(
#     "/Users/chromatrical/CAREER/Local Linkedin DB/DataBase/USA_filtered.parquet",
#     format="parquet"
# )
# ────────────────────────────────────────────────────────────────────────────────

# 2) Connect & inspect
con = duckdb.connect()

In [42]:
# 1) File size on disk
size_bytes = os.path.getsize(PARQUET_PATH)
print(f"File size: {size_bytes/1e9:.2f} GB")

File size: 16.27 GB


In [43]:
# 2) Row & column counts
con = duckdb.connect()
n_rows = con.execute(f"SELECT COUNT(*) FROM '{PARQUET_PATH}'").fetchone()[0]
dataset = ds.dataset(str(PARQUET_PATH), format="parquet")
schema = dataset.schema
n_cols = len(schema.names)
print(f"Shape: {n_rows:,} rows × {n_cols} columns\n")

Shape: 51,352,619 rows × 62 columns



In [44]:
# 3) Schema (column names and types)
print("Schema:")
for name, pa_type in zip(schema.names, schema.types):
    print(f"  • {name}: {pa_type}")
print()

Schema:
  • Full name: string
  • Industry: string
  • Job title: string
  • Sub Role: string
  • Industry 2: string
  • Emails: string
  • Mobile: string
  • Phone numbers: string
  • Company Name: string
  • Company Industry: string
  • Company Website: string
  • Company Size: string
  • Company Founded: string
  • Location: string
  • Locality: string
  • Metro: string
  • Region: string
  • Skills: string
  • First Name: string
  • Middle Initial: string
  • Middle Name: string
  • Last Name: string
  • Birth Year: string
  • Birth Date: string
  • Gender: string
  • LinkedIn Url: string
  • LinkedIn Username: string
  • Facebook Url: string
  • Facebook Username: string
  • Twitter Url: string
  • Twitter Username: string
  • Github Url: string
  • Github Username: string
  • Company Linkedin Url: string
  • Company Facebook Url: string
  • Company Twitter Url: string
  • Company Location Name: string
  • Company Location Locality: string
  • Company Location Metro: string
  • Co

In [45]:
# 4) Partitioning layout (if any)
print("Partitioning:", dataset.partitioning, "\n")

Partitioning: <pyarrow._dataset.DirectoryPartitioning object at 0x126cc19d0> 



In [46]:
# 5) Parquet metadata: row-group counts & compression codecs
pf = pq.ParquetFile(str(PARQUET_PATH))
print(f"Row groups: {pf.num_row_groups}")
for i in range(pf.num_row_groups):
    rg = pf.metadata.row_group(i)
    print(f"  RG {i}: {rg.num_rows} rows")
    for j in range(rg.num_columns):
        col = rg.column(j)
        col_name = pf.schema.names[j]
        print(f"    – {col_name}: compression={col.compression}")
print()

Row groups: 417
  RG 0: 123012 rows
    – Full name: compression=SNAPPY
    – Industry: compression=SNAPPY
    – Job title: compression=SNAPPY
    – Sub Role: compression=SNAPPY
    – Industry 2: compression=SNAPPY
    – Emails: compression=SNAPPY
    – Mobile: compression=SNAPPY
    – Phone numbers: compression=SNAPPY
    – Company Name: compression=SNAPPY
    – Company Industry: compression=SNAPPY
    – Company Website: compression=SNAPPY
    – Company Size: compression=SNAPPY
    – Company Founded: compression=SNAPPY
    – Location: compression=SNAPPY
    – Locality: compression=SNAPPY
    – Metro: compression=SNAPPY
    – Region: compression=SNAPPY
    – Skills: compression=SNAPPY
    – First Name: compression=SNAPPY
    – Middle Initial: compression=SNAPPY
    – Middle Name: compression=SNAPPY
    – Last Name: compression=SNAPPY
    – Birth Year: compression=SNAPPY
    – Birth Date: compression=SNAPPY
    – Gender: compression=SNAPPY
    – LinkedIn Url: compression=SNAPPY
    – Li

In [47]:
# 6) Preview first 5 rows
print("Preview (first 5 rows):")
preview_df = con.execute(f"SELECT * FROM '{PARQUET_PATH}' LIMIT 5").df()
print(preview_df, "\n")


Preview (first 5 rows):
             Full name                     Industry                Job title   
0       tonia guettler            consumer services           self employeed  \
1  evangeline bertloff            consumer services  chief executive officer   
2      shaun rohrbaugh           building materials              construcion   
3    charlene buchanon                 construction                     aide   
4     shawanda barrett  security and investigations      security supervisor   

  Sub Role Industry 2                                         Emails Mobile   
0     None       None                                           None   None  \
1     None       None                                           None   None   
2     None       None                             ushaun@verizon.net   None   
3     None       None                                           None   None   
4     None       None  bjaylex@hotmail.com, shawanda.barrett@drs.com   None   

  Phone numbers     

In [48]:
# 7) Missing-value profile
cols = schema.names
exprs = ", ".join(
    f"SUM(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) AS \"{c}\""
    for c in cols
)
missing_counts = con.execute(
    f"SELECT {exprs} FROM '{PARQUET_PATH}'"
).df().iloc[0]

md = pd.DataFrame({
    "column": missing_counts.index,
    "missing": missing_counts.values.astype(int)
})
md["pct_missing"] = md["missing"] / n_rows * 100
md = md.sort_values("pct_missing", ascending=False).reset_index(drop=True)
print("Missing-value summary:")
print(md, "\n")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Missing-value summary:
               column   missing  pct_missing
0          Github Url  51184112    99.671863
1     Github Username  51184077    99.671795
2      Address Line 2  50713170    98.754788
3    Twitter Username  50417015    98.178079
4         Twitter Url  50416990    98.178031
..                ...       ...          ...
57  LinkedIn Username     36468     0.071015
58          Last Name     35690     0.069500
59       LinkedIn Url     35306     0.068752
60           Location     30472     0.059339
61          Full name        16     0.000031

[62 rows x 3 columns] 



In [49]:
# 8) Cardinality (distinct counts) for categorical columns
print("Distinct value counts:")
for col in cols:
    distinct = con.execute(
        f"SELECT COUNT(DISTINCT \"{col}\") FROM '{PARQUET_PATH}'"
    ).fetchone()[0]
    print(f"  • {col}: {distinct}")
print()

Distinct value counts:
  • Full name: 25844105
  • Industry: 4456
  • Job title: 7725090
  • Sub Role: 5838
  • Industry 2: 1087
  • Emails: 23360750
  • Mobile: 2678068
  • Phone numbers: 6323282
  • Company Name: 9610051
  • Company Industry: 548
  • Company Website: 1636323
  • Company Size: 9342
  • Company Founded: 1364
  • Location: 37185
  • Locality: 21539
  • Metro: 1049
  • Region: 386


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  • Skills: 20657020
  • First Name: 1145439
  • Middle Initial: 1053
  • Middle Name: 461652
  • Last Name: 2544993
  • Birth Year: 336
  • Birth Date: 29085
  • Gender: 93
  • LinkedIn Url: 51214288
  • LinkedIn Username: 51214389
  • Facebook Url: 5910867
  • Facebook Username: 5910723
  • Twitter Url: 927632
  • Twitter Username: 927605
  • Github Url: 167549
  • Github Username: 167549
  • Company Linkedin Url: 2050671
  • Company Facebook Url: 258239
  • Company Twitter Url: 347726
  • Company Location Name: 47144
  • Company Location Locality: 33871
  • Company Location Metro: 453
  • Company Location Region: 2813
  • Company Location Geo: 40832
  • Company Location Street Address: 835572
  • Company Location Address Line 2: 16808
  • Company Location Postal Code: 44587
  • Company Location Country: 295
  • Company Location Continent: 31
  • Last Updated: 52
  • Start Date: 10776
  • Job Summary: 5069362
  • Location Country: 6772
  • Location Continent: 1917
  • Street Address:

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  • Summary: 28947876
  • Countries: 92114
  • Interests: 2743440



In [53]:
import pyarrow.types as patypes

# 9) Descriptive stats for numeric columns
numeric_cols = [
    c for c, t in zip(schema.names, schema.types)
    if patypes.is_integer(t) or patypes.is_floating(t)
]
if numeric_cols:
    agg_expr = ", ".join(
        f"MIN(\"{c}\") AS {c}_min, "
        f"MAX(\"{c}\") AS {c}_max, "
        f"AVG(\"{c}\") AS {c}_mean, "
        f"STDDEV(\"{c}\") AS {c}_std"
        for c in numeric_cols
    )
    stats = con.execute(f"SELECT {agg_expr} FROM '{PARQUET_PATH}'").df().T
    stats.columns = ["value"]
    print("Numeric descriptive statistics:")
    print(stats, "\n")

In [54]:
# 10) Estimated memory footprint if loaded into pandas
sample = preview_df  # reuse the 5-row sample
mem_per_row = sample.memory_usage(deep=True).sum() / len(sample)
est_total_mem = mem_per_row * n_rows
print(f"Estimated in-memory size: {est_total_mem/1e9:.2f} GB\n")

Estimated in-memory size: 147.05 GB



In [55]:
# 11) Date range checks for timestamp/date fields
date_cols = [c for c, t in zip(schema.names, schema.types) if t in ("timestamp[us]", "date")]
if date_cols:
    for col in date_cols:
        mn, mx = con.execute(
            f"SELECT MIN(\"{col}\"), MAX(\"{col}\") FROM '{PARQUET_PATH}'"
        ).fetchone()
        print(f"  • {col} → range: {mn} to {mx}")
    print()

In [56]:
# 12) Integrity check: duplicate primary‐key candidates
# (you may replace 'id' with your actual key column)
pk = "id"
if pk in cols:
    dup_count = con.execute(
        f"SELECT COUNT(*) - COUNT(DISTINCT \"{pk}\") FROM '{PARQUET_PATH}'"
    ).fetchone()[0]
    print(f"Duplicate '{pk}' values: {dup_count}\n")

In [57]:
# 13) Simple anomaly scan for numeric columns (IQR outliers)
print("Numeric outlier counts (IQR method):")
for col in numeric_cols:
    q1, q3 = con.execute(
        f"SELECT PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY \"{col}\"), "
        f"PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY \"{col}\") "
        f"FROM '{PARQUET_PATH}'"
    ).fetchone()
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = con.execute(
        f"SELECT COUNT(*) FROM '{PARQUET_PATH}' "
        f"WHERE \"{col}\" < {low} OR \"{col}\" > {high}"
    ).fetchone()[0]
    print(f"  • {col}: {outliers} outliers (outside [{low:.2f}, {high:.2f}])")

Numeric outlier counts (IQR method):


In [59]:
# Read only 5 rows, across however many row-groups it needs
table = dataset.head(5, columns=ds.schema.names)
df = table.to_pandas()

from IPython.display import HTML, display
html = df.to_html(index=False)
display(HTML(f"<div style='max-height:200px;overflow:auto'>{html}</div>"))

AttributeError: module 'pyarrow.dataset' has no attribute 'schema'

In [ ]:
# 4) Compute missing‐counts in one pass
cols = sample.columns.tolist()
exprs = ", ".join(
    f"SUM(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) AS \"{c}\""
    for c in cols
)
missing_series = con.execute(
    f"SELECT {exprs} FROM '{PARQUET_PATH}'"
).df().iloc[0]

# 5) Build a DataFrame of percentages
md = pd.DataFrame({
    'column': missing_series.index,
    'missing': missing_series.values
})
md['pct_missing'] = md['missing'] / n_rows * 100
md = md.sort_values('pct_missing', ascending=False).reset_index(drop=True)

# 6) Plot: sorted, gradient + highlight
norm = mpl.colors.Normalize(vmin=md['pct_missing'].min(), vmax=md['pct_missing'].max())
cmap_default = plt.cm.Blues
cmap_highlight = plt.cm.Reds

colors = [
    cmap_highlight(norm(p)) if col in highlight_cols else cmap_default(norm(p))
    for col, p in zip(md['column'], md['pct_missing'])
]

plt.figure(figsize=(10, len(md)*0.25))
plt.barh(md['column'], md['pct_missing'], color=colors)
plt.xlabel('Missing Data (%)')
plt.title('Missing Data Percentage by Column')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))